# vLLM Gateway — Load Test Walkthrough

This notebook walks through three progressively heavier load scenarios against the gateway:

| Stage | Users | Duration | Purpose |
|-------|-------|----------|---------|
| Baseline | 1 | 2 min | Single-user latency floor |
| Medium | 5 | 3 min | Moderate concurrency |
| Peak | 10 | 3 min | High concurrency / KV cache pressure |

**Before starting:** ensure vLLM (`:8000`), the gateway (`:8080`), and Prometheus + Grafana are all running.  
Open Grafana at `http://localhost:3000` alongside this notebook so you can watch metrics change in real time as each cell runs.

Run cells top-to-bottom, one at a time. Each cell is independent enough to re-run if needed.

---
## Step 1 — Set working directory

We move the kernel's working directory to the repo root so that all subsequent `!` shell commands resolve paths correctly (e.g. `loadtest/locustfile.py`, `loadtest/results/`).  
`os.chdir()` affects the kernel process permanently for this session — every later `!` command inherits it.

In [1]:
import os

REPO = "/workspace/vllm-gateway-native"
os.chdir(REPO)

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

Working directory: /workspace/vllm-gateway-native
Contents: ['grafana-template.json', 'requirements-ml.txt', 'README.md', '.gitignore', '.env.example', '.env', 'vllmenv', 'scripts', 'prometheus', 'loadtest', 'grafana', 'gateway', 'data', '.git']


---
## Step 2 — Confirm services are reachable

Before sending any load, we check that both vLLM and the gateway are up and healthy.  
The gateway's `/health` endpoint simply returns `{"status": "ok"}` — no inference happens.  
vLLM's `/health` confirms the model is loaded and the inference engine is ready.

If either curl returns an error, stop here and start the relevant service.

In [3]:
print("=== Gateway health ===")
!curl -sf http://localhost:8080/health

print("\n\n=== vLLM health ===")
!curl -sf http://localhost:8000/health

print("\n\n=== Models served by vLLM ===")
!curl -sf http://localhost:8000/v1/models | python3 -m json.tool

=== Gateway health ===
{"status":"ok"}

=== vLLM health ===


=== Models served by vLLM ===
{
    "object": "list",
    "data": [
        {
            "id": "qwen2.5-3b",
            "object": "model",
            "created": 1779382007,
            "owned_by": "vllm",
            "root": "qwen2.5-3b",
            "parent": null,
            "permission": [
                {
                    "id": "modelperm-586f8a83991e4b23babfa9f21e4cda6d",
                    "object": "model_permission",
                    "created": 1779382007,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


---
## Step 3 — Fire a single manual request through the gateway

This confirms the full request path works end-to-end: gateway auth → rate limit check → proxy to vLLM → response.  
We use the `X-API-Key` header that matches the key hardcoded in `loadtest/locustfile.py` (`key-dev-123`).  
Check the `X-Request-ID` header in the response — that UUID ties every log line for this request together.

In [13]:
!curl -s -i http://localhost:8080/v1/chat/completions \
  -H "X-API-Key: key-dev-123" \
  -H "Content-Type: application/json" \
  -d '{"model": "qwen2.5-3b","messages": [{"role": "user", "content": "What is the capital of France?"}],"max_tokens": 64 }' 

HTTP/1.1 200 OK
date: Thu, 21 May 2026 16:48:28 GMT
server: uvicorn
x-request-id: cc0dd635-69f6-4235-9f56-5a7335f85248
content-length: 341
content-type: application/json

{"id":"cmpl-f4bcc2118ecf44bfbf318b77c4524e8f","object":"chat.completion","created":1779382109,"model":"qwen2.5-3b","choices":[{"index":0,"message":{"role":"assistant","content":"The capital of France is Paris."},"logprobs":null,"finish_reason":"stop","stop_reason":null}],"usage":{"prompt_tokens":36,"total_tokens":44,"completion_tokens":8}}

---
## Step 4 — Inspect the Prometheus metrics endpoint

The gateway exposes a `/metrics` endpoint that Prometheus scrapes every 15 seconds.  
We can query it directly to see the raw counters and histograms before any load is applied — these are our baseline readings.  
After the load tests run, coming back here will show how the values changed.

In [14]:
print("=== Gateway Prometheus metrics (filtered to gateway_ prefix) ===")
!curl -sf http://localhost:8080/metrics | grep -E '^(gateway_|# HELP gateway_|# TYPE gateway_)'

=== Gateway Prometheus metrics (filtered to gateway_ prefix) ===


---
## Step 5 — Review the locustfile

The locustfile defines what each simulated user does.  
The `StandardUser` class has three task types with different weights — this controls the mix of traffic:

- `short_prompt` — weight 3 (60% of requests) — quick factual questions, `max_tokens=128`
- `long_prompt` — weight 1 (20%) — detailed technical questions, `max_tokens=512`
- `streaming_request` — weight 1 (20%) — short question via SSE stream

Each user also waits 1–3 seconds between tasks (`wait_time = between(1, 3)`), simulating human pacing.

In [ ]:
!cat loadtest/locustfile.py

---
## Step 6 — Create the results directory

Locust writes CSV files (per-endpoint stats + failure counts) and an HTML report after each run.  
We create the output directory now so the `--csv` and `--html` flags don't fail.

In [ ]:
!mkdir -p loadtest/results
print("Results directory ready:", os.path.abspath("loadtest/results"))

---
## Step 7 — Baseline run: 1 user, 2 minutes

We start with a single simulated user to establish a baseline.  
With only one user and 1–3 s think time between tasks, the request rate will be low — roughly 0.3–0.5 req/s.  
This lets vLLM process each request with no queue, giving us the minimum achievable latency under this config.

Locust runs in `--headless` mode — it prints a live stats table to stdout every few seconds and a summary when it finishes.  
The `--csv` flag writes three files: `c1_stats.csv`, `c1_stats_history.csv`, `c1_failures.csv`.  
Grafana will also update in real time as requests flow through the gateway.

In [ ]:
!vllmenv/bin/locust \
  -f loadtest/locustfile.py \
  --headless \
  --host http://localhost:8080 \
  --users 1 \
  --spawn-rate 1 \
  --run-time 2m \
  --csv loadtest/results/c1 \
  --html loadtest/results/report_1x.html

---
## Step 8 — Read the baseline CSV results

We load the stats CSV that Locust just wrote and display it as a DataFrame.  
The key columns to look at are: `Request Count`, `Failure Count`, `Median Response Time`, `95%ile`, `Requests/s`.

In [ ]:
import pandas as pd

df1 = pd.read_csv("loadtest/results/c1_stats.csv")
cols = ["Name", "Request Count", "Failure Count", "Median Response Time",
        "95%", "99%", "Average Response Time", "Requests/s"]
available = [c for c in cols if c in df1.columns]
print("=== Baseline (1 user, 2 min) ===")
print(df1[available].to_string(index=False))

---
## Step 9 — Cooldown: 15 seconds

We pause briefly between runs to let any in-flight requests complete and let the KV cache drain back to a calm state.  
In Grafana you should see the active-requests gauge drop to zero and the request rate fall to nil during this window.

In [ ]:
import time

print("Cooling down...")
for i in range(15, 0, -1):
    print(f"  {i}s remaining", end="\r")
    time.sleep(1)
print("Done — ready for next run.")

---
## Step 10 — Medium load: 5 users, 3 minutes

We bring up five concurrent users with a spawn rate of 2 users/second — so all five are active within the first 3 seconds.  
With 5 users each firing ~0.3–0.5 req/s, we get roughly 1.5–2.5 req/s total.  
Some requests will queue behind others in vLLM's scheduler. Watch the vLLM dashboard in Grafana for `num_requests_waiting`.

Long-prompt requests from multiple users may begin competing for KV cache blocks.

In [ ]:
!vllmenv/bin/locust \
  -f loadtest/locustfile.py \
  --headless \
  --host http://localhost:8080 \
  --users 5 \
  --spawn-rate 2 \
  --run-time 3m \
  --csv loadtest/results/c5 \
  --html loadtest/results/report_5x.html

---
## Step 11 — Read the medium-load results

We load the 5-user stats and compare them against the 1-user baseline side by side.

In [ ]:
df5 = pd.read_csv("loadtest/results/c5_stats.csv")

compare_cols = ["Name", "Request Count", "Failure Count",
                "Median Response Time", "95%", "Requests/s"]

a1 = [c for c in compare_cols if c in df1.columns]
a5 = [c for c in compare_cols if c in df5.columns]

print("=== Baseline (1 user, 2 min) ===")
print(df1[a1].to_string(index=False))

print("\n=== Medium load (5 users, 3 min) ===")
print(df5[a5].to_string(index=False))

---
## Step 12 — Cooldown: 15 seconds

Second cooldown before the heaviest run. Same as before — let the system settle.

In [ ]:
print("Cooling down...")
for i in range(15, 0, -1):
    print(f"  {i}s remaining", end="\r")
    time.sleep(1)
print("Done — ready for peak run.")

---
## Step 13 — Peak load: 10 users, 3 minutes

Ten concurrent users ramp up at 3 users/second — all are active within ~4 seconds.  
At this concurrency the mix of short, long, and streaming requests all contend for the same GPU.  
Watch the following panels in Grafana as this runs:

- **Active Requests** (gateway panel) — how many requests are simultaneously in-flight through the gateway  
- **vLLM — Requests Waiting** — how deeply the vLLM scheduler queue is building  
- **GPU Cache Utilization** — KV cache block pressure (will hit higher % than the 5-user run)  
- **P95 Latency** — how tail latency responds as the queue builds

The rate limiter is set at 60 req/min per IP. With 10 users from a single machine you might begin hitting it — any 429 responses will appear in Locust's failure count.

In [ ]:
!vllmenv/bin/locust \
  -f loadtest/locustfile.py \
  --headless \
  --host http://localhost:8080 \
  --users 10 \
  --spawn-rate 3 \
  --run-time 3m \
  --csv loadtest/results/c10 \
  --html loadtest/results/report_10x.html

---
## Step 14 — Side-by-side comparison: all three runs

We aggregate the `Aggregated` row from each run's stats CSV (that row sums across all task types) to get a single per-run summary.

In [ ]:
df10 = pd.read_csv("loadtest/results/c10_stats.csv")

summary_cols = ["Name", "Request Count", "Failure Count",
                "Median Response Time", "95%", "99%",
                "Average Response Time", "Min Response Time",
                "Max Response Time", "Requests/s", "Failures/s"]

labels = ["1 user (2 min)", "5 users (3 min)", "10 users (3 min)"]
frames = [df1, df5, df10]

rows = []
for label, df in zip(labels, frames):
    agg = df[df["Name"] == "Aggregated"]
    if agg.empty:
        agg = df.tail(1)
    row = agg.copy()
    row.insert(0, "Run", label)
    rows.append(row)

combined = pd.concat(rows, ignore_index=True)
available = ["Run"] + [c for c in summary_cols if c in combined.columns and c != "Name"]
print(combined[available].to_string(index=False))

---
## Step 15 — Check for failures and their causes

If any requests failed (rate-limited 429s, timeouts, 5xx errors), the failures CSV captures them.  
Each row shows the error message that Locust recorded — which maps directly to what `resp.failure(...)` was called with in the locustfile.

In [ ]:
for label, prefix in [("1 user", "c1"), ("5 users", "c5"), ("10 users", "c10")]:
    path = f"loadtest/results/{prefix}_failures.csv"
    try:
        df_fail = pd.read_csv(path)
        if df_fail.empty:
            print(f"[{label}] No failures recorded.")
        else:
            print(f"\n=== Failures — {label} ===")
            print(df_fail.to_string(index=False))
    except FileNotFoundError:
        print(f"[{label}] Failures CSV not found at {path}")

---
## Step 16 — Pull a live Prometheus snapshot post-test

We query the gateway's `/metrics` endpoint one more time.  
The counters are cumulative since server start, so we can see the total request count across all three runs, the latency histogram bucket distribution, and the token throughput counters.

In [ ]:
print("=== Final gateway metrics (gateway_ prefix) ===")
!curl -sf http://localhost:8080/metrics | grep -E '^(gateway_|# HELP gateway_|# TYPE gateway_)'

---
## Step 17 — List all output files

Every Locust run wrote a stats CSV, a history CSV (stats sampled over time), a failures CSV, and an HTML report.  
Open the HTML reports in a browser for a visual breakdown — they include response time percentile graphs per task type.

In [ ]:
!ls -lh loadtest/results/

---
## Optional — Interactive Locust UI

If you want to control users and spawn rate in real time (instead of fixed headless runs), start Locust without `--headless`.  
Run the cell below, then open `http://localhost:8089` in your browser.

**Note:** this cell runs until you interrupt the kernel (`■` button or `Kernel → Interrupt`).  
Use it for exploratory testing — manually ramp users up and down while watching Grafana.

In [ ]:
# Uncomment to run the interactive Locust UI on http://localhost:8089
# Interrupt the kernel to stop it.

# !vllmenv/bin/locust \
#   -f loadtest/locustfile.py \
#   --host http://localhost:8080